# RL for LLMs: PPO from scratch, then GRPO

LLM Training Lab. Part 1 trains SmolLM2-135M with a hand-written PPO to continue IMDb review openings positively, scored by a sentiment classifier. Part 2 trains a LoRA adapter on Qwen3-0.6B-Base with GRPO on GSM8K, scored by a verifiable reward.

Run the cells in order on one T4 (Colab: Runtime > Change runtime type > T4 GPU. Kaggle: Settings > Accelerator > GPU T4 x2). Steps 1 and 2 also run on a CPU. Code cell numbers match the lab page.

<!-- llt-run-help -->
## How to run this notebook

- **Check the lab card** on the module page first. It says whether this lab needs a GPU and how long it took us.
- **Kaggle** (where we recorded most runs): use the lab page's *Open on Kaggle* button. If the lab needs a GPU, open the notebook settings and pick *Accelerator: GPU T4 x2* (not P100). Switch *Internet* on so the install cell and the downloads work. Kaggle asks you to verify a phone number once before it allows a GPU or internet.
- **Colab**: use the lab page's *Open in Colab* button. If the lab needs a GPU, pick *Runtime > Change runtime type > T4 GPU*.
- **Hugging Face**: no account or token is needed. Our checkpoints and data files download from the public repo [aimlcompanion/llm-training-lab](https://huggingface.co/aimlcompanion/llm-training-lab).
- **Run the cells in order.** Code cell numbers match the lab page. Your numbers should land close to ours, not match exactly: GPU runs are not bit-for-bit repeatable.

In [ ]:
import os, sys, platform, shutil, subprocess
from importlib.metadata import version, PackageNotFoundError
os.environ["CUDA_VISIBLE_DEVICES"] = "0"            # one GPU, set before torch loads
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
problems = []
if sys.version_info < (3, 11):
    problems.append("Python 3.11 or newer is needed (3.12 recommended). Start a current Colab or Kaggle session.")
try:
    smi = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True).stdout.strip().splitlines()
except FileNotFoundError:                                   # a CPU runtime has no nvidia-smi
    smi = []
gpu = smi[0] if smi else ""
if not gpu:
    problems.append("No GPU: steps 1, 2, 7 and 8 still run. For step 3 on, Colab: Runtime > Change runtime type > T4 GPU. Kaggle: Settings > Accelerator > GPU T4 x2.")
elif "P100" in gpu:
    problems.append("A P100 has no kernels in current PyTorch. Pick a T4.")
cache = os.path.expanduser("~/.cache")
os.makedirs(cache, exist_ok=True)
free_disk = shutil.disk_usage(cache).free / 1e9
if free_disk < 8:
    problems.append("Free at least 8 GB of disk for the model and data downloads (" + str(round(free_disk)) + " GB free).")
try:
    torch_version = version("torch")
except PackageNotFoundError:
    torch_version = "not installed"
print("Python", platform.python_version(), "| GPU", gpu or "none", "| torch", torch_version, "| disk free", round(free_disk), "GB")
for p in problems:
    print("PROBLEM", p)
print("Runtime OK" if not problems else "Read the PROBLEM lines first.")

In [ ]:
# Pinned Hugging Face stack of our recorded runs. torch and numpy stay as the runtime ships them.
%pip uninstall -y -q torchao
%pip install -q transformers==5.22.1 trl==1.17.1 peft==0.21.2 accelerate==1.15.0 datasets==5.1.0 tokenizers==0.23.3 huggingface_hub==2.4.0 math-verify==0.9.0

## Step 1. Derive the policy gradient and write REINFORCE with a baseline (NumPy, CPU)

Code cell 1 on the lab page.

In [ ]:
import numpy as np

# One prompt, four possible replies. The policy is a softmax over four logits; each reply has a fixed reward.
reward = np.array([0.0, 1.0, 2.0, 5.0])
theta = np.zeros(4)

def probs(theta):
    e = np.exp(theta - theta.max())
    return e / e.sum()

def exact_grad(theta, reward):
    # d/dtheta of E[reward] = sum over replies of p(y) * reward(y) * grad log p(y); grad log p(y) = onehot(y) - p
    p = probs(theta)
    return p * (reward - p @ reward)

def reinforce(theta, reward, n, baseline, rng):
    # REINFORCE: sample n replies, weight grad log p(y) by (reward - baseline), average
    p = probs(theta)
    ys = rng.choice(len(p), size=n, p=p)
    return ((reward[ys] - baseline)[:, None] * (np.eye(len(p))[ys] - p)).mean(0)

rng = np.random.default_rng(0)
print("exact gradient", np.round(exact_grad(theta, reward), 4))
for shift in [0.0, 100.0]:
    r = reward + shift
    for name, b in [("no baseline", 0.0), ("baseline = mean reward", probs(theta) @ r)]:
        est = np.array([reinforce(theta, r, 8, b, rng) for _ in range(2000)])
        print("rewards +" + str(int(shift)), "|", name, "| mean of 2,000 estimates", np.round(est.mean(0), 3),
              "| spread (std, summed over the 4 logits)", round(float(est.std(0).sum()), 3))

# Train with 8 samples per step and the batch mean as the baseline, as GRPO does with a group
theta = np.zeros(4)
for step in range(201):
    p = probs(theta)
    if step % 50 == 0:
        print("step", step, "| p(best reply)", round(float(p[3]), 3), "| expected reward", round(float(p @ reward), 3))
    ys = rng.choice(4, size=8, p=p)
    adv = reward[ys] - reward[ys].mean()
    theta += 0.5 * (adv[:, None] * (np.eye(4)[ys] - p)).mean(0)

## Step 2. Write the PPO pieces and test them (PyTorch, CPU)

Code cell 2 on the lab page.

In [ ]:
import torch

def token_logps(logits, ids):
    # log-prob of each chosen token: log_softmax over the vocabulary, then pick the id
    return torch.log_softmax(logits.float(), -1).gather(-1, ids.unsqueeze(-1)).squeeze(-1)

def whiten(x):
    # zero mean, unit variance over the whole batch (the advantage normalization of most PPO code)
    return (x - x.mean()) * torch.rsqrt(x.var(unbiased=False) + 1e-8)

def gae(rewards, values, gamma=1.0, lam=0.95):
    # rewards, values: [batch, response tokens]. values[:, t] estimates the return from token t on; it is 0 after the last token.
    adv = torch.zeros_like(rewards)
    last = torch.zeros_like(rewards[:, 0])
    for t in reversed(range(rewards.shape[1])):
        next_v = values[:, t + 1] if t + 1 < rewards.shape[1] else torch.zeros_like(last)
        delta = rewards[:, t] + gamma * next_v - values[:, t]       # one-step TD error
        last = delta + gamma * lam * last
        adv[:, t] = last
    return adv, adv + values                                      # advantages, and returns for the value loss

def ppo_policy_loss(new_lp, old_lp, adv, eps=0.2):
    # clipped surrogate: take the worse (larger) of the two losses, so a big ratio earns no extra credit
    ratio = torch.exp(new_lp - old_lp)
    loss = torch.max(-adv * ratio, -adv * torch.clamp(ratio, 1 - eps, 1 + eps)).mean()
    clipped = ((ratio > 1 + eps) & (adv > 0)) | ((ratio < 1 - eps) & (adv < 0))   # tokens whose gradient the clip zeroes
    return loss, clipped.float().mean()

def value_loss(new_v, old_v, returns, eps=0.2):
    v_clip = old_v + torch.clamp(new_v - old_v, -eps, eps)
    return 0.5 * torch.max((new_v - returns) ** 2, (v_clip - returns) ** 2).mean()

def kl_k1(lp, ref_lp):
    return lp - ref_lp                      # log(pi / ref) at the sampled token

def kl_k3(lp, ref_lp):
    d = ref_lp - lp
    return torch.exp(d) - d - 1             # never negative, same expectation as k1 under pi

def shaped_rewards(score, lp, ref_lp, kl_coef):
    # PPO-RLHF reward: a KL penalty on every token, plus the reward model's score on the last token
    r = -kl_coef * kl_k1(lp, ref_lp)
    r[:, -1] += score
    return r

# Tests on small tensors
torch.manual_seed(0)
rew, val = torch.randn(3, 5), torch.randn(3, 5)
a1, ret1 = gae(rew, val, gamma=1.0, lam=1.0)
assert torch.allclose(a1, rew.flip(1).cumsum(1).flip(1) - val, atol=1e-6), "lam = 1: advantage = reward-to-go - value"
a0, _ = gae(rew, val, gamma=1.0, lam=0.0)
assert torch.allclose(a0, rew + torch.cat([val[:, 1:], torch.zeros(3, 1)], 1) - val, atol=1e-6), "lam = 0: one-step TD error"
assert torch.allclose(ret1, rew.flip(1).cumsum(1).flip(1), atol=1e-6), "lam = 1: return = reward-to-go"

lp = torch.zeros(4, requires_grad=True)
loss, cf = ppo_policy_loss(lp, torch.zeros(4), torch.tensor([1.0, -1.0, 1.0, -1.0]))
assert abs(loss.item()) < 1e-7 and cf.item() == 0, "ratio 1: loss = -mean(adv), nothing clipped"
for r, a in [(1.5, 1.0), (0.5, 1.0), (1.5, -1.0), (0.5, -1.0)]:
    x = torch.tensor([float(torch.log(torch.tensor(r)))], requires_grad=True)
    loss, cf = ppo_policy_loss(x, torch.zeros(1), torch.tensor([a]))
    loss.backward()
    print("ratio", r, "| advantage", a, "| loss", round(loss.item(), 3), "| gradient", round(x.grad.item(), 3), "| clip zeroes the gradient", bool(cf.item()))

# KL estimators: sample tokens from pi, compare with the exact KL(pi || ref) over a 50-token vocabulary
logits = torch.randn(50)
for name, noise in [("policy near the reference", 0.3), ("policy far from it", 1.5)]:
    pi, ref = torch.softmax(logits, 0), torch.softmax(logits + noise * torch.randn(50), 0)
    exact = (pi * (pi / ref).log()).sum().item()
    ids = torch.multinomial(pi, 100000, replacement=True)
    k1, k3 = kl_k1(pi.log()[ids], ref.log()[ids]), kl_k3(pi.log()[ids], ref.log()[ids])
    print(name, "| exact KL", round(exact, 4), "| k1 mean", round(k1.mean().item(), 4), "std", round(k1.std().item(), 3),
          "min", round(k1.min().item(), 3), "| k3 mean", round(k3.mean().item(), 4), "std", round(k3.std().item(), 3), "min", round(k3.min().item(), 4))

r = shaped_rewards(torch.tensor([2.0]), torch.tensor([[-1.0, -2.0, -0.5]]), torch.tensor([[-1.2, -1.5, -0.5]]), 0.05)
print("shaped rewards", [round(v, 3) for v in r[0].tolist()])
print("All PPO pieces pass their tests")

## Step 3. Load the four PPO models and score a rollout (GPU from here)

Code cell 3 on the lab page.

In [ ]:
import os
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")      # Kaggle has 2 GPUs; this lab uses one
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
import copy, gc, json, random, time
import torch
from datasets import load_dataset
from transformers import AutoModel, AutoModelForCausalLM, AutoModelForSequenceClassification, AutoTokenizer

DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
POLICY, RM = "HuggingFaceTB/SmolLM2-135M", "lvwerra/distilbert-imdb"
P_LEN, R_LEN = 8, 32                 # every prompt is 8 tokens, every response exactly 32
tok = AutoTokenizer.from_pretrained(POLICY)
rm_tok = AutoTokenizer.from_pretrained(RM)

def gb():
    return round(torch.cuda.memory_allocated() / 1e9, 3) if DEV.type == "cuda" else 0.0

class ValueModel(torch.nn.Module):
    # the same transformer body as the policy, with one number per position instead of a vocabulary
    def __init__(self):
        super().__init__()
        self.body = AutoModel.from_pretrained(POLICY, dtype=torch.float32)
        self.head = torch.nn.Linear(self.body.config.hidden_size, 1)
        torch.nn.init.zeros_(self.head.weight)
        torch.nn.init.zeros_(self.head.bias)
    def forward(self, ids):
        h = self.body(input_ids=ids).last_hidden_state
        return self.head(h).squeeze(-1)

def load_models():
    mem = {"start": gb()}
    policy = AutoModelForCausalLM.from_pretrained(POLICY, dtype=torch.float32).to(DEV)        # trained, fp32 master weights
    mem["policy"] = gb()
    ref = AutoModelForCausalLM.from_pretrained(POLICY, dtype=torch.float16 if DEV.type == "cuda" else torch.float32).to(DEV).eval()
    mem["reference"] = gb()
    value = ValueModel().to(DEV)                                                               # trained, fp32
    mem["value"] = gb()
    rm = AutoModelForSequenceClassification.from_pretrained(RM).to(DEV).eval()                 # frozen sentiment classifier
    mem["reward"] = gb()
    for m in (ref, rm):
        m.requires_grad_(False)
    return policy, ref, value, rm, mem

# Prompts: the first 8 tokens of IMDb reviews. Train prompts from the train split, held-out prompts from the test split.
imdb = load_dataset("stanfordnlp/imdb")
def prefixes(split, n, seed):
    out = []
    for text in imdb[split].shuffle(seed=seed)["text"]:
        ids = tok(text[:300].replace("<br />", " ")).input_ids
        if len(ids) >= 2 * P_LEN:
            out.append(ids[:P_LEN])
        if len(out) == n:
            return torch.tensor(out)
train_prompts, eval_prompts = prefixes("train", 12800, 0), prefixes("test", 128, 0)

@torch.no_grad()
def score(seqs, rm):
    # reward = the classifier's log-odds that prompt + response is a positive review
    enc = rm_tok(tok.batch_decode(seqs, skip_special_tokens=True), return_tensors="pt", padding=True, truncation=True, max_length=128).to(DEV)
    logits = rm(**enc).logits.float()
    return logits[:, 1] - logits[:, 0]

@torch.no_grad()
def generate(policy, prompts, seed=None):
    # fp32 sampling at temperature 1, no top-k or top-p, exactly R_LEN new tokens
    if seed is not None:
        torch.manual_seed(seed)
    prompts = prompts.to(DEV)
    return policy.generate(prompts, attention_mask=torch.ones_like(prompts), do_sample=True, temperature=1.0, top_k=0, top_p=1.0,
                           max_new_tokens=R_LEN, min_new_tokens=R_LEN, pad_token_id=tok.eos_token_id)

def response_logps(model, seqs):
    # log-probs and entropy of the R_LEN response tokens; the logit at position t predicts token t + 1
    logits = model(input_ids=seqs).logits[:, P_LEN - 1:-1].float()
    lp = torch.log_softmax(logits, -1)
    return lp.gather(-1, seqs[:, P_LEN:, None]).squeeze(-1), -(lp.exp() * lp).sum(-1)

policy, ref, value, rm, mem = load_models()
seqs = generate(policy, eval_prompts[:8], seed=0)
s = score(seqs, rm)
for i in range(3):
    print(round(s[i].item(), 2), "|", repr(tok.decode(seqs[i], skip_special_tokens=True)))
print("memory GB after each model", mem, "| response tokens per row", seqs.shape[1] - P_LEN)
print("train prompts", len(train_prompts), "| held-out prompts", len(eval_prompts), "| mean score of 8 samples", round(s.mean().item(), 3))

## Step 4. Train PPO on IMDb prefixes (RUN_PPO = False downloads our policy)

Code cell 4 on the lab page.

In [ ]:
RUN_PPO = False      # False: download our trained policy and only evaluate it. True: train it here (28 min on our shared A100; not yet timed on a T4)
PPO = dict(iters=200, batch=64, epochs=4, minibatch=16, lr=1e-5, kl_coef=0.05, clip=0.2, vf_clip=0.2, gamma=1.0, lam=0.95, seed=0)
AMP = DEV.type == "cuda"     # fp16 autocast on the GPU; the trained weights stay fp32

def autocast():
    return torch.autocast("cuda", dtype=torch.float16, enabled=AMP)

@torch.no_grad()
def evaluate(policy, ref, rm, n=128):
    # held-out prompts, one sample each with a fixed seed: mean score, mean KL to the reference, share of distinct tokens
    rng = (torch.random.get_rng_state(), torch.cuda.get_rng_state() if DEV.type == "cuda" else None)
    seqs = generate(policy, eval_prompts[:n], seed=1234)
    torch.random.set_rng_state(rng[0])
    if rng[1] is not None:
        torch.cuda.set_rng_state(rng[1])     # put the training RNG back, so evaluating does not change the training samples
    with autocast():
        lp, _ = response_logps(policy, seqs)
        rlp, _ = response_logps(ref, seqs)
    resp = seqs[:, P_LEN:]
    distinct = sum(len(set(r.tolist())) for r in resp) / resp.numel()
    return dict(score=round(score(seqs, rm).mean().item(), 4), kl=round((lp - rlp).sum(1).mean().item(), 4),
                distinct=round(distinct, 4), samples=tok.batch_decode(seqs[:4], skip_special_tokens=True))

def train_ppo(cfg, log_every=10, eval_every=25):
    torch.manual_seed(cfg["seed"]); random.seed(cfg["seed"])
    policy = AutoModelForCausalLM.from_pretrained(POLICY, dtype=torch.float32).to(DEV)
    value = ValueModel().to(DEV)
    opt_p = torch.optim.AdamW(policy.parameters(), lr=cfg["lr"], eps=1e-5)
    opt_v = torch.optim.AdamW(value.parameters(), lr=cfg["lr"], eps=1e-5)
    sc_p = torch.amp.GradScaler("cuda", init_scale=2048, enabled=AMP)
    sc_v = torch.amp.GradScaler("cuda", init_scale=2048, enabled=AMP)
    if DEV.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
    hist, evals, t0 = [], [], time.time()
    for it in range(cfg["iters"] + 1):
        if it % eval_every == 0 or it == cfg["iters"]:
            policy.eval()
            evals.append(dict(iter=it, **evaluate(policy, ref, rm)))
            print("eval", it, {k: v for k, v in evals[-1].items() if k != "samples"}, flush=True)
        if it == cfg["iters"]:
            break
        # 1. Rollout: sample responses, score them, record old log-probs, reference log-probs and values
        policy.eval()
        prompts = train_prompts[it * cfg["batch"]:(it + 1) * cfg["batch"]]
        seqs = generate(policy, prompts)
        with torch.no_grad(), autocast():
            sc = score(seqs, rm)
            old_lp, _ = response_logps(policy, seqs)
            ref_lp, _ = response_logps(ref, seqs)
            old_v = value(seqs)[:, P_LEN - 1:-1].float()     # value of the state before each response token
        # 2. Rewards and advantages
        rewards = shaped_rewards(sc, old_lp, ref_lp, cfg["kl_coef"])
        adv, ret = gae(rewards, old_v, cfg["gamma"], cfg["lam"])
        adv = whiten(adv)
        # 3. Several epochs of minibatch updates on this one batch
        policy.train(); stats = []
        for epoch in range(cfg["epochs"]):
            for mb in torch.randperm(len(seqs), device=DEV).split(cfg["minibatch"]):
                with autocast():
                    new_lp, ent = response_logps(policy, seqs[mb])
                    pl, cf = ppo_policy_loss(new_lp, old_lp[mb], adv[mb], cfg["clip"])
                opt_p.zero_grad(set_to_none=True)
                sc_p.scale(pl).backward(); sc_p.unscale_(opt_p)
                torch.nn.utils.clip_grad_norm_(policy.parameters(), 1.0); sc_p.step(opt_p); sc_p.update()
                with autocast():
                    vl = value_loss(value(seqs[mb])[:, P_LEN - 1:-1].float(), old_v[mb], ret[mb], cfg["vf_clip"])
                opt_v.zero_grad(set_to_none=True)
                sc_v.scale(vl).backward(); sc_v.unscale_(opt_v)
                torch.nn.utils.clip_grad_norm_(value.parameters(), 1.0); sc_v.step(opt_v); sc_v.update()
                with torch.no_grad():
                    stats.append([cf.item(), kl_k3(new_lp.detach(), old_lp[mb]).mean().item(), vl.item(), ent.mean().item(),
                                  torch.exp(new_lp.detach() - old_lp[mb]).max().item()])
        s = torch.tensor(stats).mean(0).tolist()
        hist.append(dict(iter=it + 1, score=round(sc.mean().item(), 4), kl=round((old_lp - ref_lp).sum(1).mean().item(), 4),
                         clipfrac=round(s[0], 4), approxKL=round(s[1], 5), valueLoss=round(s[2], 4), entropy=round(s[3], 4),
                         ratioMax=round(max(r[4] for r in stats), 3), sec=round(time.time() - t0, 1)))
        if (it + 1) % log_every == 0:
            print(hist[-1], flush=True)
    peak = round(torch.cuda.max_memory_allocated() / 1e9, 2) if DEV.type == "cuda" else None
    del value, opt_p, opt_v
    gc.collect()
    if DEV.type == "cuda":
        torch.cuda.empty_cache()
    return policy, dict(cfg=cfg, hist=hist, evals=evals, peakGB=peak, sec=round(time.time() - t0, 1))

runs = {}
if RUN_PPO:
    trained, runs["ppo-kl0.05-s0"] = train_ppo(PPO)
else:
    trained = AutoModelForCausalLM.from_pretrained("aimlcompanion/llm-training-lab", subfolder="lltRL/ppo-kl0.05-s0", dtype=torch.float32).to(DEV)
start, end = evaluate(policy, ref, rm), evaluate(trained, ref, rm)
print("held-out score", start["score"], "->", end["score"], "| KL to the reference", start["kl"], "->", end["kl"],
      "| distinct tokens", start["distinct"], "->", end["distinct"])
for t in end["samples"][:3]:
    print(repr(t))

## Step 5. Sweep the KL coefficient (RUN_EXPERIMENT = True trains two more policies)

Code cell 5 on the lab page.

In [ ]:
RUN_EXPERIMENT = False   # True trains two more policies, 31 and 35 min on our shared A100. False: read our runs below

def summary(name, r):
    e = r["evals"][-1]
    print(name, "| held-out score", e["score"], "| KL", e["kl"], "| distinct tokens", e["distinct"], "| last clip fraction", r["hist"][-1]["clipfrac"])
    print("   ", repr(e["samples"][0][:160]))

if RUN_EXPERIMENT:
    for kl in [0.0, 0.2]:
        _, runs["ppo-kl" + str(kl) + "-s0"] = train_ppo(dict(PPO, kl_coef=kl), log_every=50)
        summary("kl_coef " + str(kl), runs["ppo-kl" + str(kl) + "-s0"])
        gc.collect(); torch.cuda.empty_cache()

## Step 6. Remove the clip, cut the epochs (RUN_EXPERIMENT = True trains two more policies)

Code cell 6 on the lab page.

In [ ]:
RUN_EXPERIMENT = False   # True trains two more policies, 35 and 15 min on our shared A100. False: read our runs below

if RUN_EXPERIMENT:
    for name, cfg in [("noclip", dict(PPO, clip=1e6, vf_clip=1e6)),     # no clip on the policy or the value
                      ("epochs1", dict(PPO, epochs=1))]:                # one pass over each batch instead of four
        _, runs["ppo-" + name + "-s0"] = train_ppo(cfg, log_every=50)
        summary(name, runs["ppo-" + name + "-s0"])
        h = runs["ppo-" + name + "-s0"]["hist"]
        print("    largest ratio in any minibatch", max(x["ratioMax"] for x in h), "| mean approx KL per iteration", round(sum(x["approxKL"] for x in h) / len(h), 5))
        gc.collect(); torch.cuda.empty_cache()

## Step 7. Write a verifiable reward and group advantages (CPU)

Code cell 7 on the lab page.

In [ ]:
import os, re
import torch
from math_verify import parse, verify

NL = chr(10)
STOPS = [NL + "Question", NL + "[Question]"]       # a base model answers, then starts a new question; stop there
TIMEOUT = None if os.name == "nt" else 5            # math-verify's timeout uses signals, which Windows lacks

def prompt(q):
    return "Question: " + q + NL + "Answer:"

def cut(text):
    # the reply up to the first stop string
    ends = [text.find(s) for s in STOPS if s in text]
    return text[:min(ends)] if ends else text

def final_answer(reply):
    # the text after the last "####" or "answer is", else the last number in the reply
    reply = cut(reply)
    marked = re.findall("(?:####|answer is)([^" + NL + "]*)", reply)
    if marked:
        return marked[-1]
    nums = re.findall("-?[0-9][0-9,]*(?:[.][0-9]+)?", reply)
    return nums[-1] if nums else ""

def correct(reply, gold):
    # 1.0 if math-verify finds the reply's final answer equal to the gold number, else 0.0
    pred = parse(final_answer(reply).replace(",", ""), parsing_timeout=TIMEOUT)
    return float(bool(pred) and verify(parse(gold, parsing_timeout=TIMEOUT), pred, timeout_seconds=TIMEOUT))

def group_advantages(rewards, G, scale=True):
    # GRPO: compare each reply with the other replies to the same prompt; no value model
    r = rewards.view(-1, G)
    adv = r - r.mean(1, keepdim=True)
    if scale:                                        # divide by the group's std (Bessel-corrected, as TRL does)
        adv = adv / (r.std(1, keepdim=True) + 1e-4)
    return adv.reshape(-1)

def rloo_advantages(rewards, G):
    # RLOO: the baseline of each reply is the mean of the other G - 1 replies
    r = rewards.view(-1, G)
    return (r - (r.sum(1, keepdim=True) - r) / (G - 1)).reshape(-1)

gold = "18"
for reply in [" Janet sells 16 - 3 - 4 = 9 eggs. 9 * 2 = 18. The answer is 18." + NL + "Question: What is 2 + 2?",
              " She makes 9 * 2 = $18.00 a day." + NL + NL + "Question: next",
              " 16 - 3 - 4 = 9 eggs, so she makes 9 * 2 = 18 dollars. #### 18",
              " The answer is 16.",
              " She sells the rest at the market."]:
    print(correct(reply, gold), "| final answer", repr(final_answer(reply)), "|", repr(cut(reply)[:60]))

rewards = torch.tensor([1., 0., 0., 1., 0., 0., 0., 1.,     # prompt A: 3 of 8 correct
                        1., 1., 1., 1., 1., 1., 1., 1.,     # prompt B: all correct
                        0., 0., 0., 0., 0., 0., 0., 0.,     # prompt C: none correct
                        0., 0., 0., 0., 0., 0., 0., 1.])    # prompt D: 1 of 8 correct
for name, adv in [("GRPO, scaled by std", group_advantages(rewards, 8)), ("Dr. GRPO, not scaled", group_advantages(rewards, 8, scale=False)),
                  ("RLOO", rloo_advantages(rewards, 8))]:
    a = adv.view(4, 8)
    print(name, "| correct reply in A, D:", round(a[0, 0].item(), 3), round(a[3, 7].item(), 3), "| wrong reply in A, D:",
          round(a[0, 1].item(), 3), round(a[3, 0].item(), 3), "| groups B and C all zero:", bool((a[1:3] == 0).all()))
zero = (rewards.view(-1, 8).std(1) == 0).float().mean().item()
print("share of groups with zero spread, which give no gradient:", zero)

## Step 8. Write the GRPO loss and match GRPOTrainer on one batch (CPU)

Code cell 8 on the lab page.

In [ ]:
import tempfile
import torch
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import GRPOConfig, GRPOTrainer

def grpo_loss(new_lp, old_lp, adv, mask, eps=0.2, eps_high=None, beta=0.0, ref_lp=None, loss_type="dapo", max_len=None, denom=None):
    # new_lp, old_lp, ref_lp, mask: [replies, completion tokens]; adv: one number per reply.
    # denom: the normalizer of the whole minibatch when you accumulate micro-batches (None = this batch alone)
    hi = eps if eps_high is None else eps_high                       # DAPO's clip-higher raises only the upper bound
    ratio = torch.exp(new_lp - old_lp)
    per_tok = -torch.min(ratio * adv[:, None], torch.clamp(ratio, 1 - eps, 1 + hi) * adv[:, None])
    if beta:
        per_tok = per_tok + beta * kl_k3(new_lp, ref_lp)            # k3 KL to the reference, added per token
    if loss_type == "grpo":                                          # mean over each reply's tokens, then over replies
        return ((per_tok * mask).sum(1) / mask.sum(1).clamp(min=1)).sum() / (denom or mask.shape[0])
    if loss_type == "dr_grpo":                                       # divide by a constant, replies x max length
        return (per_tok * mask).sum() / (denom or mask.shape[0] * max_len)
    return (per_tok * mask).sum() / (denom or mask.sum().clamp(min=1))   # "dapo": every token in the batch counts once

def completion_logps(model, prompt_ids, prompt_mask, completion_ids, completion_mask):
    ids = torch.cat([prompt_ids, completion_ids], 1)
    att = torch.cat([prompt_mask, completion_mask], 1)
    # logits_to_keep: compute the vocabulary logits only for the last C + 1 positions, the ones that predict the reply
    logits = model(input_ids=ids, attention_mask=att, logits_to_keep=completion_ids.shape[1] + 1).logits[:, :-1].float()
    return torch.log_softmax(logits, -1).gather(-1, completion_ids[..., None]).squeeze(-1)

# One fixed batch: 2 prompts x 4 replies, left-padded prompts, right-padded replies, as GRPOTrainer builds them
torch.manual_seed(0)
SMALL = "HuggingFaceTB/SmolLM2-135M"            # the loss does not depend on the model, so a small one keeps this on a CPU
gtok = AutoTokenizer.from_pretrained(SMALL, padding_side="left")
gtok.pad_token = gtok.pad_token or gtok.eos_token
cpu_policy = AutoModelForCausalLM.from_pretrained(SMALL, dtype=torch.float32).eval()
enc = gtok(["Question: What is 6 x 7?" + NL + "Answer:"] * 4 + ["Question: Add 15 and 27." + NL + "Answer:"] * 4, return_tensors="pt", padding=True)
completion_ids = torch.randint(100, 5000, (8, 12))
completion_mask = (torch.arange(12)[None] < torch.tensor([12, 9, 12, 5, 7, 12, 3, 10])[:, None]).long()
rewards = torch.tensor([1., 0., 0., 1., 0., 0., 0., 1.])
adv = group_advantages(rewards, 4)
with torch.no_grad():
    lp = completion_logps(cpu_policy, enc.input_ids, enc.attention_mask, completion_ids, completion_mask)
old_lp = lp + 0.3 * torch.randn_like(lp)          # pretend the sampling policy differed a little, so the clip matters
ref_lp = lp + 0.2 * torch.randn_like(lp)

ds = Dataset.from_dict({"prompt": ["x"] * 8})
for loss_type, beta, eps_high in [("dapo", 0.0, None), ("grpo", 0.04, None), ("dr_grpo", 0.0, 0.28)]:
    args = GRPOConfig(output_dir=tempfile.mkdtemp(), use_cpu=True, per_device_train_batch_size=8, num_generations=4,
                      max_completion_length=12, loss_type=loss_type, beta=beta, epsilon_high=eps_high, bf16=False, report_to="none")
    trainer = GRPOTrainer(model=cpu_policy, reward_funcs=lambda completions, **kw: [0.0] * len(completions), args=args,
                          train_dataset=ds, processing_class=gtok)
    trainer.model.eval()                            # eval mode: TRL skips the gradient-accumulation scaling
    inputs = dict(prompt_ids=enc.input_ids, prompt_mask=enc.attention_mask, completion_ids=completion_ids, completion_mask=completion_mask,
                  advantages=adv, old_per_token_logps=old_lp, ref_per_token_logps=ref_lp, num_items_in_batch=completion_mask.sum())
    with torch.no_grad():
        theirs = trainer._compute_loss(trainer.model, inputs).item()
        new_lp = completion_logps(cpu_policy, enc.input_ids, enc.attention_mask, completion_ids, completion_mask)
        ours = grpo_loss(new_lp, old_lp, adv, completion_mask, 0.2, eps_high, beta, ref_lp, loss_type, 12).item()
    print(loss_type, "| beta", beta, "| epsilon_high", eps_high, "| ours", round(ours, 6), "| GRPOTrainer", round(theirs, 6), "| difference", abs(ours - theirs))
print("TRL GRPOConfig defaults: loss_type", GRPOConfig.loss_type, "| beta", GRPOConfig.beta, "| num_generations", GRPOConfig.num_generations,
      "| epsilon", GRPOConfig.epsilon, "| scale_rewards", GRPOConfig.scale_rewards, "| learning_rate", GRPOConfig.learning_rate)

## Step 9. Load the GRPO policy and measure the base model

Code cell 9 on the lab page.

In [ ]:
import math
from peft import LoraConfig, PeftModel, get_peft_model
from transformers import GenerationConfig

for name in ["policy", "ref", "value", "rm", "trained"]:     # free the PPO models before loading the GRPO policy
    globals().pop(name, None)
gc.collect()
if DEV.type == "cuda":
    torch.cuda.empty_cache()

GMODEL = "Qwen/Qwen3-0.6B-Base"
N_TEST, MAX_NEW = 200, 256
PASS_AT_K = False     # True: also sample 8 replies per test question for pass@k (12 min more on our shared A100)
qtok = AutoTokenizer.from_pretrained(GMODEL, padding_side="left")
gsm = load_dataset("openai/gsm8k", "main")
test = [dict(q=e["question"], gold=e["answer"].split("####")[-1].strip().replace(",", "")) for e in gsm["test"].select(range(N_TEST))]
train = [dict(q=e["question"], gold=e["answer"].split("####")[-1].strip().replace(",", "")) for e in gsm["train"].shuffle(seed=0)]

def load_policy(adapter=None):
    # frozen fp16 base + fp32 LoRA r16 on every linear layer; adapter: a saved LoRA to continue from
    base = AutoModelForCausalLM.from_pretrained(GMODEL, dtype=torch.float16 if DEV.type == "cuda" else torch.float32).to(DEV)
    base.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    base.enable_input_require_grads()
    if adapter:
        model = PeftModel.from_pretrained(base, "aimlcompanion/llm-training-lab", subfolder=adapter, is_trainable=True)
    else:
        model = get_peft_model(base, LoraConfig(r=16, lora_alpha=32, lora_dropout=0.0, target_modules="all-linear", task_type="CAUSAL_LM"))
    for p in model.parameters():
        if p.requires_grad:
            p.data = p.data.float()
    return model

def sample(model, prompts, temperature=1.0, batch=64):
    # completions as token-id lists. A reply ends at a stop string (pad after it is dropped) or at a real end-of-text token (kept).
    greedy = temperature == 0
    cfg = GenerationConfig(max_new_tokens=MAX_NEW, do_sample=not greedy, temperature=None if greedy else temperature,
                           top_k=None if greedy else 0, top_p=None if greedy else 1.0, stop_strings=STOPS,
                           pad_token_id=qtok.pad_token_id, eos_token_id=qtok.eos_token_id)
    model.eval(); model.config.use_cache = True
    out = []
    for i in range(0, len(prompts), batch):
        enc = qtok(prompts[i:i + batch], return_tensors="pt", padding=True).to(DEV)
        with torch.no_grad(), autocast():
            g = model.generate(**enc, generation_config=cfg, tokenizer=qtok)
        for row in g[:, enc.input_ids.shape[1]:].tolist():
            n = row.index(qtok.eos_token_id) if qtok.eos_token_id in row else len(row)
            stopped = any(s in qtok.decode(row[:n]) for s in STOPS)
            out.append(row[:n] if stopped or n == len(row) else row[:n + 1])
    model.config.use_cache = False
    return out

def pass_at_k(n, c, k):
    # unbiased estimate of the chance that at least one of k samples is right, from n samples with c right
    return 1.0 if n - c < k else 1.0 - math.comb(n - c, k) / math.comb(n, k)

def evaluate_gsm(model, k_samples=0):
    t = time.time()
    replies = [qtok.decode(r) for r in sample(model, [prompt(x["q"]) for x in test], temperature=0)]
    right = [int(correct(r, x["gold"])) for r, x in zip(replies, test)]
    res = dict(greedy=round(sum(right) / len(test), 4), right=right,
               meanTokens=round(sum(len(qtok(cut(r)).input_ids) for r in replies) / len(replies), 1), replies=replies[:20])
    if k_samples:
        rows = sample(model, [prompt(x["q"]) for x in test for _ in range(k_samples)])
        c = [int(sum(correct(qtok.decode(rows[i * k_samples + j]), x["gold"]) for j in range(k_samples))) for i, x in enumerate(test)]
        res["nCorrect"] = c
        for k in [1, 2, 4, 8]:
            if k <= k_samples:
                res["pass@" + str(k)] = round(sum(pass_at_k(k_samples, ci, k) for ci in c) / len(test), 4)
    res["sec"] = round(time.time() - t, 1)
    return res

policy = load_policy()
print("trainable", sum(p.numel() for p in policy.parameters() if p.requires_grad), "| dtypes", {p.dtype for p in policy.parameters() if p.requires_grad},
      "| pad id = EOS id:", qtok.pad_token_id == qtok.eos_token_id)
base_eval = evaluate_gsm(policy, 8 if PASS_AT_K else 0)
print("base model on", N_TEST, "GSM8K test questions", {k: v for k, v in base_eval.items() if k != "replies"})
print(repr(base_eval["replies"][0][:300]))

## Step 10. Train GRPO by hand (START_FROM = None trains all 150 steps)

Code cell 10 on the lab page.

In [ ]:
START_FROM = "lltRL/grpo-s0-step130"   # our adapter after 130 of 150 steps; you train the last 20. None: train all 150 yourself
GRPO = dict(steps=150, prompts=4, G=8, lr=2e-5, beta=0.0, eps=0.2, eps_high=None, loss_type="dapo", adv="grpo", minibatches=2, micro=4,
            seed=0, eval_every=0, save_at=())

def train_grpo(cfg, model=None, first_step=0, tag="grpo", resume=None):
    # resume: a folder for a resume file, written every 10 steps; a dropped session restarts from it
    torch.manual_seed(cfg["seed"]); random.seed(cfg["seed"])
    model = model or load_policy()
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=cfg["lr"], weight_decay=0.0)
    scaler = torch.amp.GradScaler("cuda", init_scale=2048, enabled=AMP)
    n = cfg["prompts"] * cfg["G"]
    hist, evals, groups, t0 = [], [], [], time.time()
    state_file = os.path.join(resume, tag + "-resume.pt") if resume else None
    st = None
    if state_file and os.path.exists(state_file):
        try:
            st = torch.load(state_file, weights_only=False)
        except RuntimeError as e:                                   # a save cut off by a dropped session
            print("resume file unreadable, starting from step", first_step, "|", str(e)[:80], flush=True)
    if st:
        for p, v in zip(params, st["params"]):
            p.data.copy_(v)
        opt.load_state_dict(st["opt"]); scaler.load_state_dict(st["scaler"])
        torch.random.set_rng_state(st["rng"][0])
        if st["rng"][1] is not None:
            torch.cuda.set_rng_state(st["rng"][1])
        first_step, hist, evals, groups = st["step"], st["hist"], st["evals"], st["groups"]
        print("resumed at step", first_step, flush=True)
    if DEV.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
    for step in range(first_step, cfg["steps"]):
        ts = time.time()
        batch = train[step * cfg["prompts"]:(step + 1) * cfg["prompts"]]
        texts = [prompt(x["q"]) for x in batch for _ in range(cfg["G"])]
        # 1. Sample G replies per prompt and score them
        rows = sample(model, texts, batch=n)
        gen_sec = time.time() - ts
        rewards = torch.tensor([correct(qtok.decode(r), x["gold"]) for r, x in zip(rows, [x for x in batch for _ in range(cfg["G"])])])
        if cfg["adv"] == "rloo":
            adv = rloo_advantages(rewards, cfg["G"])
        else:
            adv = group_advantages(rewards, cfg["G"], scale=cfg["adv"] == "grpo")
        adv = adv.to(DEV)
        # 2. One padded batch: prompt left-padded, reply right-padded, mask from each reply's length
        penc = qtok(texts, return_tensors="pt", padding=True).to(DEV)
        L = max(len(r) for r in rows)
        comp = torch.full((n, L), qtok.pad_token_id, device=DEV)
        cmask = torch.zeros((n, L), dtype=torch.long, device=DEV)
        for i, r in enumerate(rows):
            comp[i, :len(r)] = torch.tensor(r, device=DEV)
            cmask[i, :len(r)] = 1
        # 3. Log-probs of the sampling policy and of the reference (the base model: LoRA switched off)
        with torch.no_grad(), autocast():
            old_lp = torch.cat([completion_logps(model, penc.input_ids[i:i + cfg["micro"]], penc.attention_mask[i:i + cfg["micro"]],
                                                 comp[i:i + cfg["micro"]], cmask[i:i + cfg["micro"]]) for i in range(0, n, cfg["micro"])])
            with model.disable_adapter():
                ref_lp = torch.cat([completion_logps(model, penc.input_ids[i:i + cfg["micro"]], penc.attention_mask[i:i + cfg["micro"]],
                                                     comp[i:i + cfg["micro"]], cmask[i:i + cfg["micro"]]) for i in range(0, n, cfg["micro"])])
        # 4. Updates: the batch is split into minibatches, one optimizer step each, so later ones see a ratio that is not 1
        model.train(); clip, ent = [], []
        for mb in torch.randperm(n, device=DEV).chunk(cfg["minibatches"]):
            m = cmask[mb]
            denom = {"grpo": len(mb), "dr_grpo": len(mb) * MAX_NEW}.get(cfg["loss_type"], m.sum().item())
            opt.zero_grad(set_to_none=True)
            for j in range(0, len(mb), cfg["micro"]):
                idx = mb[j:j + cfg["micro"]]
                with autocast():
                    new_lp = completion_logps(model, penc.input_ids[idx], penc.attention_mask[idx], comp[idx], cmask[idx])
                    loss = grpo_loss(new_lp, old_lp[idx], adv[idx], cmask[idx], cfg["eps"], cfg["eps_high"], cfg["beta"], ref_lp[idx],
                                     cfg["loss_type"], MAX_NEW, denom)
                scaler.scale(loss).backward()
                with torch.no_grad():
                    ratio = torch.exp(new_lp - old_lp[idx]); a = adv[idx][:, None]; hi = cfg["eps_high"] or cfg["eps"]
                    clipped = ((ratio > 1 + hi) & (a > 0)) | ((ratio < 1 - cfg["eps"]) & (a < 0))
                    clip.append(((clipped * cmask[idx]).sum() / cmask[idx].sum()).item())
                    ent.append((-(new_lp * cmask[idx]).sum() / cmask[idx].sum()).item())   # mean -log p of sampled tokens, an entropy estimate
            scaler.unscale_(opt)
            gnorm = torch.nn.utils.clip_grad_norm_(params, 1.0).item()
            scaler.step(opt); scaler.update()
        lens = [len(r) for r in rows]
        if step == first_step or (step + 1) % 50 == 0:     # keep two scored groups now and then, to read later
            groups.append(dict(step=step + 1, questions=[x["q"] for x in batch[:2]], gold=[x["gold"] for x in batch[:2]],
                               replies=[qtok.decode(r) for r in rows[:2 * cfg["G"]]], rewards=rewards[:2 * cfg["G"]].tolist(),
                               tokens=lens[:2 * cfg["G"]], advantages=[round(a, 4) for a in adv[:2 * cfg["G"]].tolist()]))
        hist.append(dict(step=step + 1, reward=round(rewards.mean().item(), 4), zeroStd=round((rewards.view(-1, cfg["G"]).std(1) == 0).float().mean().item(), 3),
                         meanLen=round(sum(lens) / n, 1), truncated=round(sum(1 for r in rows if len(r) >= MAX_NEW) / n, 3),
                         kl=round(((kl_k3(old_lp, ref_lp) * cmask).sum() / cmask.sum()).item(), 5), clipFrac=round(sum(clip) / len(clip), 4),
                         entropy=round(sum(ent) / len(ent), 4), gradNorm=round(gnorm, 3), genSec=round(gen_sec, 1), sec=round(time.time() - ts, 1)))
        if (step + 1) % 10 == 0 or step == first_step:
            print(hist[-1], flush=True)
        if cfg["eval_every"] and (step + 1) % cfg["eval_every"] == 0:
            evals.append(dict(step=step + 1, **evaluate_gsm(model)))
            print("eval", step + 1, {k: v for k, v in evals[-1].items() if k != "replies"}, flush=True)
        if step + 1 in cfg["save_at"]:
            model.save_pretrained(os.path.join(WORK, tag + "-step" + str(step + 1)))
        if state_file and (step + 1) % 10 == 0:
            torch.save(dict(params=[p.detach().cpu() for p in params], opt=opt.state_dict(), scaler=scaler.state_dict(),
                            rng=(torch.random.get_rng_state(), torch.cuda.get_rng_state() if DEV.type == "cuda" else None), step=step + 1, hist=hist, evals=evals, groups=groups), state_file + ".tmp")
            os.replace(state_file + ".tmp", state_file)            # write then rename, so a drop mid-save keeps the old file
    peak = round(torch.cuda.max_memory_allocated() / 1e9, 2) if DEV.type == "cuda" else None
    return model, dict(cfg={k: v for k, v in cfg.items() if k != "save_at"}, firstStep=first_step, hist=hist, evals=evals, groups=groups, peakGB=peak,
                       sec=round(time.time() - t0, 1))

WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.getcwd()
del policy; gc.collect()
if START_FROM:
    grpo_model, runs["grpo-last20"] = train_grpo(GRPO, load_policy(START_FROM), first_step=130)
else:
    grpo_model, runs["grpo-s0"] = train_grpo(GRPO)
h = runs.get("grpo-last20", runs.get("grpo-s0"))["hist"]
print("steps", h[0]["step"], "to", h[-1]["step"], "| mean reward", round(sum(x["reward"] for x in h) / len(h), 3),
      "| mean reply tokens", round(sum(x["meanLen"] for x in h) / len(h), 1), "| s per step", round(sum(x["sec"] for x in h) / len(h), 1))

## Step 11. Measure the trained policy

Code cell 11 on the lab page.

In [ ]:
after = evaluate_gsm(grpo_model, 8 if PASS_AT_K else 0)
print("after GRPO", {k: v for k, v in after.items() if k != "replies"})
print("before    ", {k: v for k, v in base_eval.items() if k != "replies"})
right = [i for i, (a, b) in enumerate(zip(base_eval["replies"], after["replies"])) if correct(b, test[i]["gold"]) and not correct(a, test[i]["gold"])]
if right:
    i = right[0]
    print("question", i, "gold", test[i]["gold"])
    print("before:", repr(cut(base_eval["replies"][i])[:300]))
    print("after: ", repr(cut(after["replies"][i])[:300]))

## Step 12. Change the group size, the KL term, the normalization and the baseline (RUN_EXPERIMENT = True)

Code cell 12 on the lab page.

In [ ]:
RUN_EXPERIMENT = False   # True trains six more 150-step policies, projected 7 to 8 h on a T4. False: read our runs below
ARMS = {"G4":      dict(GRPO, prompts=8, G=4),            # same 32 replies per step, smaller groups
        "G16":     dict(GRPO, prompts=2, G=16),           # bigger groups, fewer prompts
        "beta0.04": dict(GRPO, beta=0.04),                # KL penalty to the base model, the DeepSeekMath setting
        "seqmean": dict(GRPO, loss_type="grpo"),          # the original GRPO average: per reply, then over replies
        "drgrpo":  dict(GRPO, loss_type="dr_grpo", adv="drgrpo"),   # no std scaling, constant normalizer
        "rloo":    dict(GRPO, adv="rloo")}                # leave-one-out baseline instead of the group mean

if RUN_EXPERIMENT:
    del grpo_model; gc.collect(); torch.cuda.empty_cache()
    for name, cfg in ARMS.items():
        m, runs["grpo-" + name] = train_grpo(cfg, tag="grpo-" + name)
        runs["grpo-" + name]["final"] = evaluate_gsm(m)
        h = runs["grpo-" + name]["hist"]
        last, first = h[-50:], h[:50]
        print(name, "| reward, last 50 steps", round(sum(x["reward"] for x in last) / len(last), 3), "| reply tokens, first and last 50",
              round(sum(x["meanLen"] for x in first) / len(first), 1), round(sum(x["meanLen"] for x in last) / len(last), 1),
              "| zero-spread groups", round(sum(x["zeroStd"] for x in h) / len(h), 3), "| greedy accuracy", runs["grpo-" + name]["final"]["greedy"])
        del m; gc.collect(); torch.cuda.empty_cache()

## Step 13. Run the same training with GRPOTrainer (RUN_TRL = True)

Code cell 13 on the lab page.

In [ ]:
RUN_TRL = False     # True: 30 GRPOTrainer steps with the same settings, 12 min on our A100; not yet timed on a T4
from trl import GRPOConfig, GRPOTrainer

def gsm_reward(completions, gold, **kw):
    # TRL passes the dataset's extra columns (here: gold) to every reward function
    return [correct(c, g) for c, g in zip(completions, gold)]

if RUN_TRL:
    ds = Dataset.from_list([dict(prompt=prompt(x["q"]), gold=x["gold"]) for x in train[:30 * 4]])
    args = GRPOConfig(output_dir=os.path.join(WORK, "trl-grpo"), max_steps=30, learning_rate=GRPO["lr"], lr_scheduler_type="constant",
                      per_device_train_batch_size=8, gradient_accumulation_steps=4, num_generations=8,   # 32 replies, 4 prompts per step
                      max_completion_length=MAX_NEW, temperature=1.0, top_k=0, top_p=1.0, beta=0.0, loss_type="dapo",
                      fp16=True, bf16=False,          # bf16 defaults on; a T4 needs fp16. No stop strings: TRL does not pass the tokenizer to generate
                      model_init_kwargs=dict(dtype="float16"), gradient_checkpointing=True,
                      logging_steps=1, save_strategy="no", report_to="none", seed=0)
    trainer = GRPOTrainer(model=GMODEL, reward_funcs=gsm_reward, args=args, train_dataset=ds, processing_class=qtok,
                          peft_config=LoraConfig(r=16, lora_alpha=32, lora_dropout=0.0, target_modules="all-linear", task_type="CAUSAL_LM"))
    for p in trainer.model.parameters():
        if p.requires_grad:
            p.data = p.data.float()          # GradScaler needs fp32 trainable weights
    print("trainable dtypes", {p.dtype for p in trainer.model.parameters() if p.requires_grad})
    torch.cuda.reset_peak_memory_stats(); t = time.time()
    trainer.train()
    logs = [l for l in trainer.state.log_history if "reward" in l]
    runs["trl-grpo"] = dict(log=logs, sec=round(time.time() - t, 1), peakGB=round(torch.cuda.max_memory_allocated() / 1e9, 2))
    print("30 steps in", runs["trl-grpo"]["sec"], "s | peak GB", runs["trl-grpo"]["peakGB"], "| mean reward",
          round(sum(l["reward"] for l in logs) / len(logs), 3), "| mean completion length", round(sum(l["completions/mean_length"] for l in logs) / len(logs), 1))